# A4 Part 1: MuJoCo CartPole - Reinforcement Learning Implementation

In this section, you are going to set up the MuJoCo environment to do physics-based reinforcement learning. Make sure you have the required packages installed as listed in the `environment.yml` file.

## MuJoCo Installation

MuJoCo now is open-sourced project and can be installed via the pip package. You can install it by running the following command:

```bash
pip install mujoco
```

## `dm_control` (deep-mind control) Installation

`dm_control` is a package that provides a suite of continuous control tasks. You can install it by running the following command:

```bash
pip install dm_control
```

Make sure you have `cmake` installed on your system.

## `ffmpeg` Installation

`ffmpeg` is a package that is used to render the video of the environment. You can install it by running the following command:

If you are on linux, you can install it by running the following command:

```bash
sudo apt-get install ffmpeg 
```

If you are on macOS, you can install it by running the following command, make sure you have `brew` installed:

```bash
brew install ffmpeg
```

If you are on Windows, you can install it by running the following command:

```bash
conda install -c conda-forge ffmpeg
```

### `mujoco_tutorial.ipynb`

Before starting this part of the assignment, please make sure you have completed the `mujoco_tutorial.ipynb` notebook. This notebook will help you understand the basics of the MuJoCo environment and how to interact with it.

In [ ]:
%load_ext autoreload
%autoreload 2


# The basic mujoco wrapper.
from dm_control import mujoco
from dm_control import suite
from tqdm import tqdm

#@title Other imports and helper functions

# General
import copy
import os
import itertools
from IPython.display import clear_output
import numpy as np

# Graphics-related
import matplotlib
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML
import PIL.Image
# Internal loading of video libraries.

# Use svg backend for figure rendering
%config InlineBackend.figure_format = 'svg'

# Font sizes
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 12
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

from src.utils import display_video


## Cartpole Balance & Swingup Tasks

In the following section, we will look at the Cartpole Balance and Swingup tasks. The Cartpole Balance task is a classic control problem where the goal is to balance a pole on a cart. The Cartpole Swingup task is a more challenging version of the Cartpole Balance task where the goal is to swing the pole up and balance it on the cart. This will be broken into three tasks.

1. **Task 1:** Implement TD-Learning for Cartpole Balance Task
2. **Task 2:** Implement TD-Learning for Cartpole Swingup Task
3. **Task 3:** Implement TD-Learning using Continual learning

You will implement the functions in `td.py` and `discretize.py`, and submit the two files along with 3 generated plots from this notebook.

# Implement TD-Learning for Cartpole Balance Task

In this sub-part, you will implement the TD-Learning algorithm for the Cartpole Balance task. The Cartpole Balance task is a classic control problem where the goal is to balance a pole on a cart. The state space consists of the cart's position, the cart's velocity, the pole's angle, and the pole's angular velocity.

### Take a look at the environment

The following cell runs the Cartpole Balance task for a single episode and renders the environment. Below, you will implement the TD-Learning algorithm to interact with this environment and learn a policy for balancing the pole.

In [ ]:
# Random policy for cartpole balance task

# Load the environment
random_state = np.random.RandomState(42)
env = suite.load("cartpole", "balance", task_kwargs={"random": random_state})

# Simulate episode with random actions
duration = 5  # Seconds
frames = []
ticks = []
rewards_balance = []
observations = []

spec = env.action_spec()
time_step = env.reset()

# Simulate the environment (environment loop)
while env.physics.data.time < duration:
    action = random_state.uniform(spec.minimum, spec.maximum, spec.shape)
    time_step = env.step(action)
    camera0 = env.physics.render(camera_id=0, height=400, width=400)
    camera1 = env.physics.render(camera_id=1, height=400, width=400)
    frames.append(np.hstack((camera0, camera1)))
    rewards_balance.append(time_step.reward)
    observations.append(copy.deepcopy(time_step.observation))
    ticks.append(env.physics.data.time)


html_video = display_video(frames, framerate=1.0 / env.control_timestep())

html_video

In [ ]:
# random policy for cartpole swingup task

# Load the environment
random_state = np.random.RandomState(42)
env = suite.load("cartpole", "swingup", task_kwargs={"random": random_state})

# Simulate episode with random actions
duration = 5  # Seconds
frames = []
ticks = []
rewards_balance = []
observations = []

spec = env.action_spec()
time_step = env.reset()

# Simulate the environment (environment loop)
while env.physics.data.time < duration:
    action = random_state.uniform(spec.minimum, spec.maximum, spec.shape)
    time_step = env.step(action)
    camera0 = env.physics.render(camera_id=0, height=400, width=400)
    camera1 = env.physics.render(camera_id=1, height=400, width=400)
    frames.append(np.hstack((camera0, camera1)))
    rewards_balance.append(time_step.reward)
    observations.append(copy.deepcopy(time_step.observation))
    ticks.append(env.physics.data.time)


html_video = display_video(frames, framerate=1.0 / env.control_timestep())

html_video

You should see the cartpole environment rendering, but with random actions. The pole should fall over quickly. This is not very intelligent right? Luckily, we can do a lot better with reinforcement learning! One of the simplest algorithms is TD-Learning, which we will implement in this sub-part. Before doing that, let's do more analysis on the episode we just simulated.

Since the whole environment is simulated in MuJoCo, this means that we get access to every single state of the environment, including the observations and the rewards throughout the episode. The following cell will plot the observations and rewards throughout the episode. Later, you will use this graph to understand the MDP of the Cartpole Balance task.

In [ ]:
%matplotlib inline

def plot_episode_stats(observations: list, rewards: list, ticks: list):
    """
    Plot the episode statistics based on the observations

    Args:
        observations (list): List of observations from the environment
    """
    num_sensors = len(observations[0])

    _, ax = plt.subplots(1 + num_sensors, 1, sharex=True, figsize=(4, 8))
    ax[0].plot(ticks, rewards)
    ax[0].set_ylabel("reward")
    ax[-1].set_xlabel("time")

    for i, key in enumerate(observations[0]):
        data = np.asarray([observations[j][key] for j in range(len(observations))])
        ax[i + 1].plot(ticks, data, label=key)
        ax[i + 1].set_ylabel(key)
    plt.show()

plot_episode_stats(observations, rewards_balance, ticks)

## Task 1: TD Learning for Cartpole with quantization of state space and action space

In this task, we will implement TD Learning for the Cartpole environment. One way to handle continuous state and action spaces is by discretizing them into bins. This process, known as quantization, allows us to apply discrete reinforcement learning algorithms to continuous tasks. We will define the state and action bins, initialize the Q-table, train the agent using the TD Learning algorithm, and visualize the learning progress by plotting the total rewards obtained in each episode.

### Define the state and action bins

Since the cartpole environment receives a continuous state space, we need to discretize it into bins to make the state space and action space discrete. We will define the state and action bins for the Cartpole environment. The state bins will be used to discretize the continuous state space, while the action bins will be used to discretize the continuous action space. By doing this, we can construct the Q-table for the task.

Before doing that, let's take a look at the state and action spaces of the Cartpole environment.

In [ ]:
# take a look at the action spec
spec = env.action_spec()
spec

In [ ]:
# take a look at the first observation, action, and reward
observations[-1], action[0], rewards_balance[0]

For the states representation, we have 5 dimensional observations, three for the position, and two for velocity. For action space, we have 1-dimensional action space, which is the force applied to the cart. Let's start to quantize (discretize) the state and action space.

For the discretization, we will define a sequences of bins for each dimension of the state and action space. We will use the `np.linspace` function to create these bins. The number of bins for each dimension is a hyperparameter that you can tune to optimize the performance of the TD Learning algorithm.

### Hyperparameter of Discretization

The following bin configs are provided as a starting point. The algorithm might learn something but the quantization is not optimal. You can change these values to optimize the performance of the TD Learning algorithm.

In [ ]:
# Define custom bin configurations for each observation dimension.
# For each observation variable, specify a list of (min, max, bin_nums)
# You can adjust the range and bin count for each dimension as needed.

pos_bins_config = [
    (-1.0, 1.0, 5),    # Config for the first position component
    (-0.5, 0.5, 2),    # Config for the second position component
    (-0.3, 0.3, 7),    # Config for the third position component
]
vel_bins_config = [
    (-1.5, 1.5, 5),    # Config for the first velocity component
    (-3.0, 3.0, 7),    # Config for the second velocity component
]

# action_bins_config = (-1, 1, 4)
action_bins_config = (-1, 1, 5)

action_bins = np.linspace(*action_bins_config)

obs_bins_config = {"position": pos_bins_config, "velocity": vel_bins_config}

# Create a list of bins arrays corresponding to each observation dimension.
state_bins = {
    key: [
        np.linspace(min_val, max_val, bin_num) for min_val, max_val, bin_num in config
    ]
    for key, config in obs_bins_config.items()
}

after we define the bins, we now can start to quantize the states. Implement the functions that takes the continuous state and returns the quantized state in `src/discretize.py`

In [ ]:
from src.discretize import quantize_state


time_step = env.reset()
obs = time_step.observation

quantized_state = quantize_state(obs, state_bins)
print("Quantized State:", quantized_state)

Nice! Now we can tackle the continuous control problem with discrete reinforcement learning algorithms. Next, based on the quantized state and action spaces, we will initialize the Q-table for the Cartpole environment.

In [ ]:
from src.td import initialize_q_table

q_table_balance = initialize_q_table(state_bins, action_bins)
print("Initialized Q-Table shape:", q_table_balance.shape)

The Q-table shape will be dependent on how you setup the quantization. If you quantize the space with more bins, your q-table will be larger. Let's take a look at the total number of parameter in the Q-table.

In [ ]:
print("Total number of Q-values:", np.prod(q_table_balance.size))

Next, let's implement the TD learning algorithm based on the quantization and the q-table we created.

In [ ]:
from src.td import td_learning

# Define the environment and parameters
env = suite.load("cartpole", "balance")
num_episodes = 1000  # hyperparameter
alpha = 0.1  # hyperparameter
gamma = 0.99  # hyperparameter
epsilon = 0.1  # hyperparameter

# Train the agent
q_table_balance, rewards_balance = td_learning(
    env, num_episodes, alpha, gamma, epsilon, state_bins, action_bins
)

after training, we can plot the reward progression of the learner as it learns to balance the pole on the cart.

**Submission:** Submit the following reward progression plot `td-balance.png` to the Gradescope

In [ ]:
# Plot the rewards
# if it is weird looking, run it twice
plt.plot(rewards_balance)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("TD Learning for Balance Cartpole")
# save the figure for submission
plt.savefig("td-balance.png", dpi=200)
plt.show()

After training the agent, we should take a look at the q-table to see how the agent has learned to balance the pole on the cart. The following cell will displace the total number of elements in the q-table and the total number of non-zero elements in the q-table.

In [ ]:
def inspect_q_table(q_table: np.ndarray):
    """
    Inspect the Q-table for characteristics of the learned values.

    Args:
        q_table (np.ndarray): Q-values for state-action pairs.
    """
    total_elements = np.prod(q_table.shape)
    non_zero_elements = np.sum(np.isclose(q_table, 0))
    total_non_zero_elements = total_elements - non_zero_elements

    print(f"Total number of Q-values: {total_elements}")
    print(f"Number of non-zero Q-values: {total_non_zero_elements}")
    print(
        f"Percentage of non-zero Q-values: {100 * total_non_zero_elements / total_elements:.2f}%"
    )


inspect_q_table(q_table_balance)

In [ ]:
from src.td import greedy_policy
from src.utils import run_episode

env = suite.load("cartpole", "balance")

# Define the greedy policy
greedy_policy_fn = greedy_policy(q_table_balance)

# Example usage: get the action for a given state
example_state = quantize_state(env.reset().observation, state_bins)
action = greedy_policy_fn(example_state)
print(f"Greedy action for the example state: {action}")

In [ ]:
# Run an episode using the greedy policy
total_reward, frames, stats = run_episode(
    env, greedy_policy_fn, state_bins, action_bins
)

plot_episode_stats(*stats)

# Display the video of the episode
html_video = display_video(frames, framerate=50)
print(f"Total reward received during the episode: {total_reward}")
html_video

## Task 2: Train the cartpole swing up tasks

In previous section, you have trained the balance task. The swing up tasks is more difficult than the balance task since the cart needs to first swing up the pole, and then keep it balance. Let's train the swing up task with the same TD learning algorithm!

In [ ]:
from src.td import td_learning

# Define the environment and parameters
env = suite.load("cartpole", "swingup")
num_episodes = 1000  # hyperparameter
alpha = 0.1  # hyperparameter
gamma = 0.99  # hyperparameter
epsilon = 0.1  # hyperparameter

# Train the agent
q_table_swingup, rewards_swingup = td_learning(
    env, num_episodes, alpha, gamma, epsilon, state_bins, action_bins
)

**Submission:** Submit the following reward progression plot `td-swingup.png` to the Gradescope

In [ ]:
# Plot the rewards

### if it is weird looking, run it twice ###
plt.plot(rewards_swingup)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("TD Learning for Swingup Cartpole")

# save the figure for submission
plt.savefig("td-swingup.png", dpi=200)
plt.show()

In [ ]:
inspect_q_table(q_table_swingup)

In [ ]:
env = suite.load("cartpole", "swingup")

# Define the greedy policy
greedy_policy_fn = greedy_policy(q_table_swingup)

# Run an episode using the greedy policy
total_reward, frames, stats = run_episode(
    env, greedy_policy_fn, state_bins, action_bins
)


plot_episode_stats(*stats)

# Display the video of the episode
html_video = display_video(frames, framerate=50)
print(f"Total reward received during the episode: {total_reward}")
html_video

## Task 3: Continual Learning between Tasks

In the previous sections, you have trained the balance and swingup tasks separately, each starting from an empty Q-table. In this part, you will train swingup starting from the Q-table already learned on balance. This is a form of continual learning: the agent first learns to balance the pole, then builds on that knowledge to learn the swing-up.

The transfer is done by passing the balance Q-table to `td_learning` as the starting table, instead of initializing a new one. Run the cells below and compare the reward curve with your earlier swingup run.

In [ ]:
# Define the environment and parameters
env = suite.load("cartpole", "swingup")
num_episodes = 1000  # hyperparameter
alpha = 0.1  # hyperparameter
gamma = 0.99  # hyperparameter
epsilon = 0.1  # hyperparameter

# Train the agent with initial knowledge from the balance task
q_table_swingup_transfer, rewards_swingup_transfer = td_learning(
    env,
    num_episodes,
    alpha,
    gamma,
    epsilon,
    state_bins,
    action_bins,
    q_table=q_table_balance,  # reusing the q_table from the balance task
)

In [ ]:
# Plot the rewards
# if it is weird looking, run it twice
plt.plot(rewards_swingup_transfer)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("TD Learning for Transfer Swingup Cartpole from Balance")

plt.savefig("td_learning_transfer_swingup_cartpole.png")
plt.show()

In [ ]:
inspect_q_table(q_table_swingup_transfer)

In [ ]:
env = suite.load("cartpole", "swingup")

# Define the greedy policy
greedy_policy_fn = greedy_policy(q_table_swingup_transfer)

# Run an episode using the greedy policy
total_reward, frames, stats = run_episode(
    env, greedy_policy_fn, state_bins, action_bins
)


plot_episode_stats(*stats)

# Display the video of the episode
html_video = display_video(frames, framerate=50)
print(f"Total reward received during the episode: {total_reward}")
html_video

# Submission

Required files for submission:

`td.py`: Python file containing the implementation of the TD Learning algorithm.

`discretize.py`: Python file containing the implementation of the discretize function.

`td-balance.png`: Reward progression plot for the Cartpole Balance task.

`td-swingup.png`: Reward progression plot for the Cartpole Swingup task.

`td_learning_transfer_swingup_cartpole.png`: Reward progression for Cartpole transfer task.